In [ ]:
import pandas as _hex_pandas
import datetime as _hex_datetime
import json as _hex_json

In [ ]:
hex_scheduled = _hex_json.loads("false")

In [ ]:
hex_user_email = _hex_json.loads("\"example-user@example.com\"")

In [ ]:
hex_user_attributes = _hex_json.loads("{}")

In [ ]:
hex_run_context = _hex_json.loads("\"logic\"")

In [ ]:
hex_timezone = _hex_json.loads("\"UTC\"")

In [ ]:
hex_project_id = _hex_json.loads("\"019cd7cc-aa9d-7004-acde-a1214e1396d6\"")

In [ ]:
hex_project_name = _hex_json.loads("\"Sehetna Forecasting Model\"")

In [ ]:
hex_status = _hex_json.loads("\"\"")

In [ ]:
hex_categories = _hex_json.loads("[]")

In [ ]:
hex_color_palette = _hex_json.loads("[\"#4C78A8\",\"#F58518\",\"#E45756\",\"#72B7B2\",\"#54A24B\",\"#EECA3B\",\"#B279A2\",\"#FF9DA6\",\"#9D755D\",\"#BAB0AC\"]")

In [ ]:
!uv pip install torch accelerate --quiet

In [ ]:
# import jinja2
# raw_query = """
#     select *
#     from "25_countries_main.csv" as data_df
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
import pandas as pd
import numpy as np

FEATURES = [
    'temp_squared', 'pm25_ugm3', 'heat_wave_days', 'month_sin',
    'temp_precip_interaction', 'aqi_pm', 'temp_anomaly_celsius',
    'pollution_vulnerability', 'flood_indicator', 'healthcare_access_index',
    'distance_to_equator', 'pm25_change_rate', 'pm25_ugm3_lag_1w', 'month_cos',
    'temp_change_rate', 'temperature_celsius', 'gdp_per_capita_usd',
    'quarter', 'spatial_lag_pm25', 'is_northern',
    'is_tropical', 'spatial_lag_temp', 'food_security_index',
]

TARGETS = [
    'respiratory_disease_rate', 'cardio_mortality_rate',
    'vector_disease_risk_score', 'waterborne_disease_incidents',
    'heat_related_admissions',
]

SEQ_LEN = 52
HORIZON_LEN = 12
NUM_FEATURES = len(FEATURES)
NUM_TARGETS = len(TARGETS)
ALL_COLS = FEATURES + TARGETS

df = data_df.copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country_code', 'date']).reset_index(drop=True)

# --- Direct computations ---
df['temp_squared'] = df['temperature_celsius'] ** 2
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['quarter'] = (df['month'] - 1) // 3 + 1
df['temp_precip_interaction'] = df['temperature_celsius'] * df['precipitation_mm']
df['pollution_vulnerability'] = df['pm25_ugm3'] / (df['healthcare_access_index'] + 1e-6)
df['distance_to_equator'] = df['latitude'].abs()
df['is_northern'] = (df['latitude'] > 0).astype(int)
df['is_tropical'] = (df['latitude'].abs() < 23.5).astype(int)

# --- Per-country temporal features ---
df['pm25_change_rate'] = df.groupby('country_code')['pm25_ugm3'].diff()
df['temp_change_rate'] = df.groupby('country_code')['temperature_celsius'].diff()
df['pm25_ugm3_lag_1w'] = df.groupby('country_code')['pm25_ugm3'].shift(1)

# --- Spatial lag (leave-one-out mean per week) ---
for col, new_col in [('pm25_ugm3', 'spatial_lag_pm25'), ('temperature_celsius', 'spatial_lag_temp')]:
    week_sum = df.groupby('date')[col].transform('sum')
    week_count = df.groupby('date')[col].transform('count')
    df[new_col] = (week_sum - df[col]) / (week_count - 1)

# --- Backfill NaN from diff/shift (first row per country only) ---
for col in ['pm25_change_rate', 'temp_change_rate', 'pm25_ugm3_lag_1w']:
    df[col] = df.groupby('country_code')[col].bfill()

# --- Validate ---
missing_feats = set(FEATURES) - set(df.columns)
missing_tgts = set(TARGETS) - set(df.columns)
assert not missing_feats, f"Missing features: {missing_feats}"
assert not missing_tgts, f"Missing targets: {missing_tgts}"

nan_counts = df[ALL_COLS].isna().sum()
nan_cols = nan_counts[nan_counts > 0]
assert nan_cols.empty, f"NaN found:\n{nan_cols}"

df_engineered = df
print(f"Shape: {df_engineered.shape[0]} rows × {df_engineered.shape[1]} cols")
print(f"Features: {NUM_FEATURES} | Targets: {NUM_TARGETS}")
print(f"Countries: {df_engineered['country_code'].nunique()} | Date range: {df['date'].min().date()} → {df['date'].max().date()}")

Shape: 14100 rows × 46 cols
Features: 23 | Targets: 5
Countries: 25 | Date range: 2015-01-04 → 2025-10-19


In [ ]:
from sklearn.preprocessing import RobustScaler

train_frames, val_frames, test_frames = [], [], []
scalers = {}

for cc, grp in df_engineered.groupby('country_code'):
    grp = grp.sort_values('date').reset_index(drop=True)
    n = len(grp)

    train_end = int(n * 0.80)
    val_end = int(n * 0.90)

    train_part = grp.iloc[:train_end].copy()
    val_part = grp.iloc[train_end:val_end].copy()
    test_part = grp.iloc[val_end:].copy()

    # Fit scaler on train only — RobustScaler uses median/IQR, preserves peaks
    scaler = RobustScaler()
    scaler.fit(train_part[ALL_COLS])
    scalers[cc] = scaler

    train_part[ALL_COLS] = scaler.transform(train_part[ALL_COLS])
    val_part[ALL_COLS] = scaler.transform(val_part[ALL_COLS])
    test_part[ALL_COLS] = scaler.transform(test_part[ALL_COLS])

    train_frames.append(train_part)
    val_frames.append(val_part)
    test_frames.append(test_part)

train_df = pd.concat(train_frames).reset_index(drop=True)
val_df = pd.concat(val_frames).reset_index(drop=True)
test_df = pd.concat(test_frames).reset_index(drop=True)

print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows | Test: {len(test_df)} rows")
print(f"Per-country split: ~{len(train_df)//25} train, ~{len(val_df)//25} val, ~{len(test_df)//25} test weeks")
print(f"Scalers fitted for {len(scalers)} countries")

Train: 11275 rows | Val: 1400 rows | Test: 1425 rows
Per-country split: ~451 train, ~56 val, ~57 test weeks
Scalers fitted for 25 countries


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class TimeSeriesWindowDataset(Dataset):
    """Sliding window dataset for training (windows fully within split)."""

    def __init__(self, df, features, targets, seq_len, horizon_len):
        self.windows = []
        self.labels = []
        all_cols = features + targets
        target_indices = list(range(len(features), len(all_cols)))

        for _, grp in df.groupby('country_code'):
            grp = grp.sort_values('date').reset_index(drop=True)
            values = grp[all_cols].values.astype(np.float32)
            n = len(values)
            for i in range(n - seq_len - horizon_len + 1):
                x = values[i : i + seq_len]                                     # [seq_len, 28]
                y = values[i + seq_len : i + seq_len + horizon_len][:, target_indices]  # [horizon, 5]
                self.windows.append(x)
                self.labels.append(y)

        self.windows = np.array(self.windows)
        self.labels = np.array(self.labels)

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return torch.tensor(self.windows[idx]), torch.tensor(self.labels[idx])


def build_windows_with_context(current_df, prior_df, features, targets, seq_len, horizon_len):
    """Build windows for val/test, allowing context to reach into prior split."""
    windows, labels = [], []
    all_cols = features + targets
    target_indices = list(range(len(features), len(all_cols)))

    for cc in current_df['country_code'].unique():
        prior_grp = prior_df[prior_df['country_code'] == cc].sort_values('date')
        curr_grp = current_df[current_df['country_code'] == cc].sort_values('date')

        context = prior_grp.tail(seq_len)
        combined = pd.concat([context, curr_grp]).reset_index(drop=True)
        values = combined[all_cols].values.astype(np.float32)
        context_len = len(context)

        for i in range(len(combined) - seq_len - horizon_len + 1):
            label_start = i + seq_len
            if label_start < context_len:
                continue
            x = values[i : i + seq_len]
            y = values[label_start : label_start + horizon_len][:, target_indices]
            if len(y) == horizon_len:
                windows.append(x)
                labels.append(y)

    return np.array(windows, dtype=np.float32), np.array(labels, dtype=np.float32)


class ArrayDataset(Dataset):
    def __init__(self, windows, labels):
        self.windows = windows
        self.labels = labels

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return torch.tensor(self.windows[idx]), torch.tensor(self.labels[idx])


# --- Build datasets ---
train_dataset = TimeSeriesWindowDataset(train_df, FEATURES, TARGETS, SEQ_LEN, HORIZON_LEN)

val_windows, val_labels = build_windows_with_context(
    val_df, train_df, FEATURES, TARGETS, SEQ_LEN, HORIZON_LEN
)
test_windows, test_labels = build_windows_with_context(
    test_df, pd.concat([train_df, val_df]), FEATURES, TARGETS, SEQ_LEN, HORIZON_LEN
)

val_dataset = ArrayDataset(val_windows, val_labels)
test_dataset = ArrayDataset(test_windows, test_labels)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train windows: {len(train_dataset)} | Val windows: {len(val_dataset)} | Test windows: {len(test_dataset)}")
print(f"Window shape: X=[{SEQ_LEN}, {NUM_FEATURES + NUM_TARGETS}] → y=[{HORIZON_LEN}, {NUM_TARGETS}]")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
import torch.nn as nn
from transformers import PatchTSTConfig, PatchTSTModel


# =============================================================================
# TEMPORAL ATTENTION POOL — learns which patches (time segments) matter most
# =============================================================================
class TemporalAttentionPool(nn.Module):
    def __init__(self, d_model, num_heads=2):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, x):
        b = x.size(0)
        query = self.pool_query.expand(b, -1, -1)
        attn_out, _ = self.attn(query, x, x)
        return attn_out.squeeze(1)


# =============================================================================
# ENHANCED PATCHTST — backbone + custom prediction head
# =============================================================================
class EnhancedPatchTST(nn.Module):
    def __init__(self, config, num_targets, horizon_len,
                 channel_proj_dim=32, mixer_dim=256, residual_dim=128,
                 temporal_attn_heads=2, dropout=0.2):
        super().__init__()
        self.num_targets = num_targets
        self.horizon_len = horizon_len
        self.num_channels = config.num_input_channels
        self.d_model = config.d_model

        # --- PatchTST backbone (Transformer encoder) ---
        self.backbone = PatchTSTModel(config)

        # --- Custom prediction head ---
        # 1) Temporal attention: pool over patches per channel
        self.temporal_attn = TemporalAttentionPool(config.d_model, temporal_attn_heads)

        # 2) Per-channel projection: reduce d_model → compact representation
        self.channel_proj = nn.Sequential(
            nn.Linear(config.d_model, channel_proj_dim),
            nn.GELU(),
        )

        # 3) Channel mixer: learn cross-channel dependencies
        mixer_input_dim = config.num_input_channels * channel_proj_dim
        self.channel_mixer = nn.Sequential(
            nn.Linear(mixer_input_dim, mixer_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # 4) Residual MLP block
        self.residual_mlp = nn.Sequential(
            nn.Linear(mixer_dim, residual_dim),
            nn.GELU(),
            nn.Linear(residual_dim, mixer_dim),
        )
        self.residual_dropout = nn.Dropout(dropout)
        self.residual_norm = nn.LayerNorm(mixer_dim)

        # 5) Output projection → [horizon_len × num_targets]
        self.output_proj = nn.Linear(mixer_dim, horizon_len * num_targets)

    def forward(self, past_values):
        backbone_out = self.backbone(past_values=past_values)
        h = backbone_out.last_hidden_state  # [batch, channels, num_patches, d_model]
        B, C, P, D = h.shape

        h = h.reshape(B * C, P, D)
        h = self.temporal_attn(h)
        h = h.reshape(B, C, D)

        h = self.channel_proj(h)
        h = h.reshape(B, -1)
        h = self.channel_mixer(h)

        residual = h
        h = self.residual_mlp(h)
        h = self.residual_dropout(h)
        h = self.residual_norm(h + residual)

        out = self.output_proj(h)
        return out.reshape(B, self.horizon_len, self.num_targets)


# =============================================================================
# COMPOSITE LOSS: 0.3*Huber + 0.5*WeightedMSE(global IQR) + 0.2*EPLoss
# =============================================================================
class CompositeLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.5, gamma=0.2, huber_delta=2.0,
                 wmse_lambda=1.5, global_median=None, global_iqr=None):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.huber = nn.HuberLoss(delta=huber_delta, reduction='mean')
        self.wmse_lambda = wmse_lambda
        # Pre-computed global stats for stable weighting (computed from train set)
        if global_median is not None:
            self.register_buffer('global_median', global_median)
            self.register_buffer('global_iqr', global_iqr)
        else:
            self.global_median = None
            self.global_iqr = None

    def weighted_mse(self, pred, target):
        if self.global_median is not None:
            median = self.global_median
            iqr = self.global_iqr
        else:
            with torch.no_grad():
                median = target.median(dim=0, keepdim=True).values
                q75 = target.quantile(0.75, dim=0, keepdim=True)
                q25 = target.quantile(0.25, dim=0, keepdim=True)
                iqr = (q75 - q25).clamp(min=1e-6)
        weights = 1.0 + self.wmse_lambda * (target - median).abs() / iqr
        return (weights.detach() * (pred - target) ** 2).mean()

    def extreme_point_loss(self, pred, target):
        if target.shape[1] < 2:
            return torch.tensor(0.0, device=pred.device)
        dy_true = target[:, 1:, :] - target[:, :-1, :]
        dy_pred = pred[:, 1:, :] - pred[:, :-1, :]
        sign_mismatch = torch.clamp(-torch.sign(dy_pred) * torch.sign(dy_true), min=0)
        magnitude = dy_true.abs()
        return (sign_mismatch * magnitude).mean()

    def forward(self, pred, target):
        return (self.alpha * self.huber(pred, target)
                + self.beta * self.weighted_mse(pred, target)
                + self.gamma * self.extreme_point_loss(pred, target))


# =============================================================================
# INSTANTIATE MODEL — wider backbone + overlapping patches
# =============================================================================
patchtst_config = PatchTSTConfig(
    num_input_channels=NUM_FEATURES + NUM_TARGETS,   # 28
    context_length=SEQ_LEN,                          # 52
    prediction_length=HORIZON_LEN,                   # 12
    patch_length=12,                                 # overlapping patches
    stride=6,                                        # 50% overlap → 7 patches
    d_model=128,
    num_attention_heads=8,
    num_hidden_layers=4,
    ffn_dim=256,
    dropout=0.2,
    head_dropout=0.2,
)

model = EnhancedPatchTST(
    config=patchtst_config,
    num_targets=NUM_TARGETS,
    horizon_len=HORIZON_LEN,
    channel_proj_dim=32,
    mixer_dim=256,
    residual_dim=128,
    temporal_attn_heads=2,
    dropout=0.2,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Patches: {(SEQ_LEN - 12) // 6 + 1} (overlapping, stride=6)")
print(f"Architecture: PatchTST backbone ({patchtst_config.num_hidden_layers}L, d={patchtst_config.d_model}, {patchtst_config.num_attention_heads}H) + Custom Head")
print(f"Input: [{SEQ_LEN}, {NUM_FEATURES + NUM_TARGETS}] → Output: [{HORIZON_LEN}, {NUM_TARGETS}]")

In [ ]:
def compute_global_target_stats(train_loader, num_targets):
    """Pre-compute median and IQR of target labels across entire training set."""
    all_targets = []
    for _, batch_y in train_loader:
        all_targets.append(batch_y)
    all_targets = torch.cat(all_targets)  # [N, horizon, targets]
    flat = all_targets.reshape(-1, num_targets)     # [N*horizon, targets]
    median = flat.median(dim=0).values              # [targets]
    q75 = flat.quantile(0.75, dim=0)
    q25 = flat.quantile(0.25, dim=0)
    iqr = (q75 - q25).clamp(min=1e-6)
    return median, iqr


def train_model(model, train_loader, val_loader, device,
                num_epochs=100, patience=25):
    """End-to-end training with composite loss, OneCycleLR, and early stopping."""
    model = model.to(device)

    # Compute global target stats for stable WeightedMSE
    global_median, global_iqr = compute_global_target_stats(train_loader, model.num_targets)
    global_median = global_median.to(device)
    global_iqr = global_iqr.to(device)

    criterion = CompositeLoss(
        alpha=0.3, beta=0.5, gamma=0.2,
        huber_delta=2.0, wmse_lambda=1.5,
        global_median=global_median, global_iqr=global_iqr,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=5e-3)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=3e-3, epochs=num_epochs,
        steps_per_epoch=len(train_loader), pct_start=0.3
    )

    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': [], 'val_r2': []}

    for epoch in range(num_epochs):
        # --- Training ---
        model.train()
        train_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item() * batch_x.size(0)

        train_loss /= len(train_loader.dataset)

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                pred = model(batch_x)
                loss = criterion(pred, batch_y)
                val_loss += loss.item() * batch_x.size(0)
                val_preds.append(pred.cpu())
                val_targets.append(batch_y.cpu())

        val_loss /= len(val_loader.dataset)
        val_preds = torch.cat(val_preds)
        val_targets = torch.cat(val_targets)

        # Flatten [windows, horizon, targets] → [windows*horizon, targets] for metrics
        vp_flat = val_preds.reshape(-1, model.num_targets)
        vt_flat = val_targets.reshape(-1, model.num_targets)

        val_mae = (vp_flat - vt_flat).abs().mean().item()
        val_rmse = ((vp_flat - vt_flat) ** 2).mean().sqrt().item()
        ss_res = ((vp_flat - vt_flat) ** 2).sum().item()
        ss_tot = ((vt_flat - vt_flat.mean(dim=0, keepdim=True)) ** 2).sum().item()
        val_r2 = 1 - ss_res / max(ss_tot, 1e-8)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)
        history['val_rmse'].append(val_rmse)
        history['val_r2'].append(val_r2)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | MAE: {val_mae:.4f} | RMSE: {val_rmse:.4f} | R²: {val_r2:.4f}")

        # --- Early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
    return model, history


# =============================================
# >>> UNCOMMENT BELOW TO START TRAINING <<<
# >>> (Long-running: ~15-20 min on GPU)  <<<
# =============================================
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device: {device}")
# model, history = train_model(model, train_loader, val_loader, device, num_epochs=100, patience=25)

In [ ]:
def mc_dropout_inference(model, data_loader, device, n_forward=50):
    """Run N stochastic forward passes with dropout active for uncertainty estimation."""
    model.to(device)
    model.train()  # keep dropout active

    all_means, all_stds, all_targets = [], [], []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            preds = torch.stack([model(batch_x).cpu() for _ in range(n_forward)])
            all_means.append(preds.mean(dim=0))
            all_stds.append(preds.std(dim=0))
            all_targets.append(batch_y)

    return torch.cat(all_means), torch.cat(all_stds), torch.cat(all_targets)


def collapse_overlapping_predictions(all_preds, all_stds, all_targets):
    """
    Collapse overlapping sliding-window predictions into final horizon.
    Used for VISUALIZATION only — not for primary evaluation.
    """
    num_seq, horizon_len, num_targets = all_preds.shape
    total_t = num_seq + horizon_len - 1

    ts_preds = {i: [] for i in range(total_t)}
    ts_vars = {i: [] for i in range(total_t)}
    ts_tgts = {i: [] for i in range(total_t)}

    for s in range(num_seq):
        for h in range(horizon_len):
            t = s + h
            ts_preds[t].append(all_preds[s, h])
            ts_vars[t].append(all_stds[s, h] ** 2)
            ts_tgts[t].append(all_targets[s, h])

    final_preds, final_stds, final_tgts = [], [], []
    for t in range(total_t - horizon_len, total_t):
        p = torch.stack(ts_preds[t])
        v = torch.stack(ts_vars[t])
        g = torch.stack(ts_tgts[t])
        final_preds.append(p.mean(dim=0))
        final_stds.append(v.mean(dim=0).sqrt())
        final_tgts.append(g.mean(dim=0))

    return torch.stack(final_preds), torch.stack(final_stds), torch.stack(final_tgts)


# =============================================
# >>> UNCOMMENT BELOW TO RUN INFERENCE <<<
# >>> (Long-running: ~45 sec on GPU)     <<<
# =============================================
# test_means, test_stds, test_targets = mc_dropout_inference(model, test_loader, device, n_forward=50)
#
# # --- PRIMARY EVALUATION: raw window-level metrics (before collapsing) ---
# from sklearn.metrics import r2_score as sk_r2
# flat_preds = test_means.reshape(-1, test_means.shape[-1]).numpy()
# flat_targets = test_targets.reshape(-1, test_targets.shape[-1]).numpy()
# print("=== Window-Level Metrics (Primary) ===")
# for i, tgt in enumerate(TARGETS):
#     mae = np.abs(flat_preds[:, i] - flat_targets[:, i]).mean()
#     rmse = np.sqrt(((flat_preds[:, i] - flat_targets[:, i]) ** 2).mean())
#     r2 = sk_r2(flat_targets[:, i], flat_preds[:, i])
#     print(f"  {tgt:>35s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
#
# # --- SECONDARY: collapse for visualization ---
# collapsed_preds, collapsed_stds, collapsed_targets = collapse_overlapping_predictions(
#     test_means, test_stds, test_targets
# )
# ci_lower = collapsed_preds - 1.96 * collapsed_stds
# ci_upper = collapsed_preds + 1.96 * collapsed_stds
# print(f"\nCollapsed shape: {collapsed_preds.shape} | 95% CI width (mean): {(ci_upper - ci_lower).mean():.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def compute_metrics(preds, targets, target_names):
    """
    Compute metrics on raw window-level predictions.
    preds/targets: [num_windows, horizon_len, num_targets] or [timesteps, num_targets]
    """
    if preds.ndim == 3:
        preds_np = preds.reshape(-1, preds.shape[-1])
        targets_np = targets.reshape(-1, targets.shape[-1])
    else:
        preds_np = preds
        targets_np = targets

    preds_np = preds_np.numpy() if hasattr(preds_np, 'numpy') else preds_np
    targets_np = targets_np.numpy() if hasattr(targets_np, 'numpy') else targets_np

    results = []
    for i, name in enumerate(target_names):
        p, t = preds_np[:, i], targets_np[:, i]

        mae = mean_absolute_error(t, p)
        rmse = np.sqrt(mean_squared_error(t, p))
        r2 = r2_score(t, p)

        threshold = np.percentile(np.abs(t), 90)
        peak_mask = np.abs(t) > threshold
        peak_mae = mean_absolute_error(t[peak_mask], p[peak_mask]) if peak_mask.sum() > 0 else np.nan

        dt, dp = np.diff(t), np.diff(p)
        dir_acc = (np.sign(dt) == np.sign(dp)).mean() * 100 if len(dt) > 0 else np.nan

        results.append({
            'Target': name,
            'MAE': round(mae, 4),
            'RMSE': round(rmse, 4),
            'R²': round(r2, 4),
            'Peak MAE': round(peak_mae, 4) if not np.isnan(peak_mae) else 'N/A',
            'Dir. Acc (%)': round(dir_acc, 2) if not np.isnan(dir_acc) else 'N/A',
        })

    return pd.DataFrame(results)


def plot_forecasts_with_ci(preds, stds, targets, target_names):
    """Plot collapsed forecasts with 95% CI bands (visualization only)."""
    n = len(target_names)
    fig, axes = plt.subplots(n, 1, figsize=(12, 3.5 * n), sharex=True)
    if n == 1:
        axes = [axes]

    t_axis = np.arange(preds.shape[0])
    for i, (ax, name) in enumerate(zip(axes, target_names)):
        p = preds[:, i].numpy() if hasattr(preds, 'numpy') else preds[:, i]
        s = stds[:, i].numpy() if hasattr(stds, 'numpy') else stds[:, i]
        a = targets[:, i].numpy() if hasattr(targets, 'numpy') else targets[:, i]

        ax.plot(t_axis, a, 'k-', lw=1.5, label='Actual')
        ax.plot(t_axis, p, 'b-', lw=1.2, label='Predicted')
        ax.fill_between(t_axis, p - 1.96 * s, p + 1.96 * s,
                         alpha=0.25, color='steelblue', label='95% CI')
        ax.set_ylabel(name, fontsize=9)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Forecast Horizon Step')
    fig.suptitle('PatchTST Forecasts with 95% Confidence Intervals', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_training_history(history):
    """Plot loss curves, validation metrics, and R²."""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

    ax1.plot(history['train_loss'], label='Train Loss', lw=1.2)
    ax1.plot(history['val_loss'], label='Val Loss', lw=1.2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Composite Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history['val_mae'], label='Val MAE', lw=1.2)
    ax2.plot(history['val_rmse'], label='Val RMSE', lw=1.2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Metric')
    ax2.set_title('Validation MAE / RMSE')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3.plot(history['val_r2'], label='Val R²', lw=1.2, color='green')
    ax3.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Baseline (mean)')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('R²')
    ax3.set_title('Validation R²')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# =============================================
# >>> UNCOMMENT BELOW TO RUN EVALUATION  <<<
# >>> (Fast: < 5 sec)                    <<<
# =============================================
# print("=== Window-Level Metrics (test set) ===")
# metrics_df = compute_metrics(test_means, test_targets, TARGETS)
# print(metrics_df.to_string(index=False))
#
# print("\n=== Training Curves ===")
# plot_training_history(history)
#
# print("\n=== Collapsed Forecast Visualization ===")
# plot_forecasts_with_ci(collapsed_preds, collapsed_stds, collapsed_targets, TARGETS)